# SWAN-SF Sliding-Window LSTM Baseline

Train an LSTM classifier on overlapping windows extracted from the SWAN-SF time-series partitions.
The goal is to establish a strong sequential baseline with class-aware evaluation (TSS, ROC-AUC, PR-AUC)
and to compare against the tabular models.

In [18]:
# Configure project paths, dependencies, and hyperparameters
from pathlib import Path
import random
import numpy as np
import sys, os

sys.path.append(os.path.abspath("../scripts"))

from data_utils import find_project_root, load_sequence_data
from preprocessing import compute_feature_stats, normalize_sequence_data
from metrics import compute_metrics, sweep_thresholds, format_metrics
from losses import FocalLoss

PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / "data" / "Cleaned SWANSF Dataset"
print("Project root:", PROJECT_ROOT)
print("Data dir:", DATA_DIR)

# Sliding-window + training hyperparameters
WINDOW_SIZE = 32  # timesteps per window (longer temporal context)
WINDOW_STRIDE = 4
MAX_POS_WINDOWS_PER_SAMPLE = 16   # limit positives per NOAA sample to reduce overfitting spikes
MAX_NEG_WINDOWS_PER_SAMPLE = 12   # cap negative windows per sample to reduce imbalance
MAX_NEG_TO_POS_RATIO = 1.5        # additional global cap after window generation
POSITIVE_WINDOW_TAIL_FRACTION = 0.5  # keep only last 50% of windows for positive samples
NEGATIVE_WINDOW_SAMPLE_FRACTION = 0.5  # randomly subsample negative windows within each sample
AGGREGATION_MODE = "max"          # use spike-aware aggregation
AGGREGATION_PERCENTILE = 0.9      # retained for percentile mode
MAX_ACCEPTABLE_FPR = 0.05         # stricter FPR constraint during tuning
THRESHOLD_GRID = np.linspace(0.2, 0.9, 29)  # finer grid for TSS sweep (step 0.025)
BATCH_SIZE = 128
NUM_EPOCHS = 25
EARLY_STOP_PATIENCE = 5
LR_SCHEDULER_PATIENCE = 2
LR_SCHEDULER_FACTOR = 0.5
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
HIDDEN_SIZE = 128
NUM_LAYERS = 2
DROPOUT = 0.3
random_seed = 42

random.seed(random_seed)
np.random.seed(random_seed)
rng = np.random.default_rng(random_seed)

Project root: /Users/chintan/Drive/Projects/solar-flare-classification
Data dir: /Users/chintan/Drive/Projects/solar-flare-classification/data/Cleaned SWANSF Dataset


In [19]:
# Import project modules and PyTorch
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

# Restrict modeling to the top-5 features identified in EDA
SELECTED_FEATURES = ("SAVNCPP", "ABSNJZH", "R_VALUE", "EPSZ", "TOTFZ")

print("Torch version:", torch.__version__)
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)
print("Num selected attributes:", len(SELECTED_FEATURES))

Torch version: 2.11.0
Using device: mps
Num selected attributes: 5


In [20]:
# Load partitions, normalize features, and create train/val/test splits
X_train_all, y_train_all, X_val_all, y_val_all, X_test_all, y_test_all = load_sequence_data(
    DATA_DIR,
    SELECTED_FEATURES,
    )

print("Train shape (concat):", X_train_all.shape)
print("Validation shape:", X_val_all.shape)
print("Test shape (concat):", X_test_all.shape)

feature_mean, feature_std = compute_feature_stats(X_train_all)
print("Feature mean (first 5):", feature_mean[:5])
print("Feature std (first 5):", feature_std[:5])

X_train_all = normalize_sequence_data(X_train_all, feature_mean, feature_std)
X_val_all = normalize_sequence_data(X_val_all, feature_mean, feature_std)
X_test_all = normalize_sequence_data(X_test_all, feature_mean, feature_std)

Train shape (concat): (77865, 60, 5)
Validation shape: (19899, 60, 5)
Test shape (concat): (331185, 60, 5)
Feature mean (first 5): [0.79975486 0.637749   0.6846829  0.2945301  0.8939753 ]
Feature std (first 5): [0.11830382 0.18934865 0.17060669 0.10897669 0.1079974 ]


In [21]:
# Sliding window utilities and PyTorch dataset wrappers
from typing import Dict, Optional


def build_window_index(
    X: np.ndarray,
    y: np.ndarray,
    window_size: int,
    stride: int,
    max_pos_windows: Optional[int] = None,
    max_neg_windows: Optional[int] = None,
    rng: Optional[np.random.Generator] = None,
 ) -> Dict[str, np.ndarray]:
    if rng is None:
        rng = np.random.default_rng()

    sample_indices = []
    window_offsets = []
    labels = []
    sample_ids = []

    for sample_idx, (seq, label) in enumerate(zip(X, y)):
        if seq.shape[0] < window_size:
            continue
        starts = list(range(0, seq.shape[0] - window_size + 1, stride))
        if not starts:
            continue

        # Restrict positive windows to the tail where flare signatures are more likely to appear.
        if label == 1 and 0 < POSITIVE_WINDOW_TAIL_FRACTION < 1.0:
            tail_count = max(1, int(len(starts) * POSITIVE_WINDOW_TAIL_FRACTION))
            starts = starts[-tail_count:]
        # Randomly subsample negative windows early to avoid over-representing quiet sequences.
        if label == 0 and 0 < NEGATIVE_WINDOW_SAMPLE_FRACTION < 1.0:
            keep_count = max(1, int(len(starts) * NEGATIVE_WINDOW_SAMPLE_FRACTION))
            keep_idx = rng.choice(len(starts), size=keep_count, replace=False)
            starts = [starts[i] for i in keep_idx]

        limit = max_pos_windows if label == 1 else max_neg_windows
        if limit is not None and len(starts) > limit:
            keep_idx = rng.choice(len(starts), size=limit, replace=False)
            starts = [starts[i] for i in keep_idx]

        sample_indices.extend([sample_idx] * len(starts))
        window_offsets.extend(starts)
        labels.extend([float(label)] * len(starts))
        sample_ids.extend([sample_idx] * len(starts))

    if not sample_indices:
        raise ValueError("No windows were generated; check window size/stride settings.")

    return {
        "sample_indices": np.array(sample_indices, dtype=np.int64),
        "window_offsets": np.array(window_offsets, dtype=np.int32),
        "labels": np.array(labels, dtype=np.float32),
        "sample_ids": np.array(sample_ids, dtype=np.int64),
    }


def rebalance_window_index(
    data: Dict[str, np.ndarray],
    max_neg_to_pos_ratio: Optional[float],
    rng: Optional[np.random.Generator] = None,
) -> Dict[str, np.ndarray]:
    if max_neg_to_pos_ratio is None:
        return data
    if rng is None:
        rng = np.random.default_rng()

    labels = data["labels"]
    pos_idx = np.where(labels == 1)[0]
    neg_idx = np.where(labels == 0)[0]

    if len(pos_idx) == 0 or len(neg_idx) == 0:
        return data

    allowed_neg = int(len(pos_idx) * max_neg_to_pos_ratio)
    if allowed_neg <= 0 or len(neg_idx) <= allowed_neg:
        return data

    keep_neg_idx = rng.choice(neg_idx, size=allowed_neg, replace=False)
    keep_idx = np.concatenate([pos_idx, keep_neg_idx])
    keep_idx.sort()

    return {
        key: value[keep_idx]
        for key, value in data.items()
    }


def summarize_index(name: str, data: Dict[str, np.ndarray]):
    total = len(data["labels"])
    pos = int(data["labels"].sum())
    neg = total - pos
    ratio = neg / pos if pos > 0 else float("inf")
    print(
        f"{name}: windows={total:,} | pos={pos} ({pos/total:.2%}) | "
        f"neg={neg} ({neg/total:.2%}) | neg/pos={ratio:.2f}"
    )


class WindowDataset(Dataset):
    def __init__(self, X: np.ndarray, metadata: Dict[str, np.ndarray], window_size: int):
        self.X = X
        self.sample_indices = metadata["sample_indices"]
        self.window_offsets = metadata["window_offsets"]
        self.labels = torch.from_numpy(metadata["labels"]).float()
        self.sample_ids = torch.from_numpy(metadata["sample_ids"]).long()
        self.window_size = window_size

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        sample_idx = self.sample_indices[idx]
        start = self.window_offsets[idx]
        seq = self.X[sample_idx]
        window = seq[start : start + self.window_size]
        window_tensor = torch.as_tensor(window, dtype=torch.float32)
        return window_tensor, self.labels[idx], self.sample_ids[idx]


def make_dataloaders(window_size=WINDOW_SIZE, stride=WINDOW_STRIDE):
    train_meta = build_window_index(
        X_train_all,
        y_train_all,
        window_size,
        stride,
        max_pos_windows=MAX_POS_WINDOWS_PER_SAMPLE,
        max_neg_windows=MAX_NEG_WINDOWS_PER_SAMPLE,
        rng=rng,
    )
    val_meta = build_window_index(
        X_val_all,
        y_val_all,
        window_size,
        stride,
        max_pos_windows=MAX_POS_WINDOWS_PER_SAMPLE,
        max_neg_windows=MAX_NEG_WINDOWS_PER_SAMPLE,
        rng=rng,
    )
    test_meta = build_window_index(
        X_test_all,
        y_test_all,
        window_size,
        stride,
        max_pos_windows=None,
        max_neg_windows=None,
        rng=rng,
    )

    train_meta = rebalance_window_index(train_meta, MAX_NEG_TO_POS_RATIO, rng=rng)
    val_meta = rebalance_window_index(val_meta, MAX_NEG_TO_POS_RATIO, rng=rng)

    summarize_index("Train", train_meta)
    summarize_index("Val", val_meta)
    summarize_index("Test", test_meta)

    train_loader = DataLoader(WindowDataset(X_train_all, train_meta, window_size), batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
    val_loader = DataLoader(WindowDataset(X_val_all, val_meta, window_size), batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
    test_loader = DataLoader(WindowDataset(X_test_all, test_meta, window_size), batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

    return train_loader, val_loader, test_loader, train_meta, val_meta, test_meta


train_loader, val_loader, test_loader, train_data, val_data, test_data = make_dataloaders()
train_loader, val_loader, test_loader

Train: windows=311,460 | pos=151492 (48.64%) | neg=159968 (51.36%) | neg/pos=1.06
Val: windows=79,596 | pos=39600 (49.75%) | neg=39996 (50.25%) | neg/pos=1.01
Test: windows=1,324,740 | pos=24936 (1.88%) | neg=1299804 (98.12%) | neg/pos=52.13


(<torch.utils.data.dataloader.DataLoader at 0x11951ba80>,
 <torch.utils.data.dataloader.DataLoader at 0x121b5d370>)

In [22]:
# Define LSTM classifier
class LSTMClassifier(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, num_layers: int, dropout: float):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=False,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        # x: (batch, seq, features)
        outputs, (h_n, _) = self.lstm(x)
        last_hidden = h_n[-1]
        out = self.dropout(last_hidden)
        logits = self.fc(out).squeeze(1)
        return logits


input_dim = len(SELECTED_FEATURES)
model = LSTMClassifier(input_dim, HIDDEN_SIZE, NUM_LAYERS, DROPOUT).to(device)
model

LSTMClassifier(
  (lstm): LSTM(5, 128, num_layers=2, batch_first=True, dropout=0.3)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=128, out_features=1, bias=True)
)

In [23]:
# Training/evaluation utilities for window-level and sample-level predictions
from collections import defaultdict


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    for batch_x, batch_y, _ in loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad()
        logits = model(batch_x)
        loss = criterion(logits, batch_y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * batch_x.size(0)

    return running_loss / len(loader.dataset)


@torch.no_grad()
def predict_probabilities(model, loader, device):
    model.eval()
    probs = []
    labels = []
    sample_ids = []

    for batch_x, batch_y, batch_sid in loader:
        batch_x = batch_x.to(device)
        logits = model(batch_x)
        batch_probs = torch.sigmoid(logits).detach().cpu().numpy()

        probs.append(batch_probs)
        labels.append(batch_y.numpy())
        sample_ids.append(batch_sid.numpy())

    if not probs:
        return np.array([]), np.array([]), np.array([])

    return np.concatenate(probs), np.concatenate(labels), np.concatenate(sample_ids)


def aggregate_sample_probabilities(y_prob_windows, y_true_windows, sample_ids, mode=AGGREGATION_MODE, percentile=AGGREGATION_PERCENTILE):
    grouped_probs = defaultdict(list)
    grouped_labels = defaultdict(list)

    for prob, label, sid in zip(y_prob_windows, y_true_windows, sample_ids):
        grouped_probs[int(sid)].append(float(prob))
        grouped_labels[int(sid)].append(int(label))

    sample_probs = []
    sample_true = []

    for sid in sorted(grouped_probs.keys()):
        probs = np.array(grouped_probs[sid], dtype=float)
        labels = np.array(grouped_labels[sid], dtype=int)

        if mode == "mean":
            agg_prob = float(np.mean(probs))
        elif mode == "percentile":
            agg_prob = float(np.quantile(probs, percentile))
        else:  # "max"
            agg_prob = float(np.max(probs))

        agg_true = int(np.max(labels))

        sample_probs.append(agg_prob)
        sample_true.append(agg_true)

    return np.array(sample_probs), np.array(sample_true)


@torch.no_grad()
def evaluate_model(model, loader, device, threshold=0.5):
    y_prob, y_true, _ = predict_probabilities(model, loader, device)
    metrics = compute_metrics(y_true, y_prob, threshold=threshold)
    return metrics


@torch.no_grad()
def evaluate_sample_level(model, loader, device, threshold=0.5, mode=AGGREGATION_MODE, percentile=AGGREGATION_PERCENTILE):
    y_prob_w, y_true_w, sample_ids = predict_probabilities(model, loader, device)
    y_prob_s, y_true_s = aggregate_sample_probabilities(y_prob_w, y_true_w, sample_ids, mode=mode, percentile=percentile)
    metrics = compute_metrics(y_true_s, y_prob_s, threshold=threshold)
    return metrics


@torch.no_grad()
def tune_threshold_sample_level(model, loader, device, thresholds=THRESHOLD_GRID, max_fpr=None, mode=AGGREGATION_MODE, percentile=AGGREGATION_PERCENTILE):
    y_prob_w, y_true_w, sample_ids = predict_probabilities(model, loader, device)
    y_prob_s, y_true_s = aggregate_sample_probabilities(y_prob_w, y_true_w, sample_ids, mode=mode, percentile=percentile)
    best, _ = sweep_thresholds(y_true_s, y_prob_s, thresholds=thresholds, max_fpr=max_fpr)
    return best


@torch.no_grad()
def tune_threshold_window_level(model, loader, device, thresholds=THRESHOLD_GRID, max_fpr=None):
    y_prob_w, y_true_w, _ = predict_probabilities(model, loader, device)
    best, _ = sweep_thresholds(y_true_w, y_prob_w, thresholds=thresholds, max_fpr=max_fpr)
    return best

In [24]:
# Configure loss and optimizer, then train with early stopping using sample-level validation metrics
pos_count = float(np.sum(train_data["labels"]))
neg_count = float(len(train_data["labels"]) - pos_count)
pos_weight_value = neg_count / pos_count if pos_count > 0 else 1.0
print(f"Training windows: pos={int(pos_count)}, neg={int(neg_count)}, pos_weight={pos_weight_value:.2f}")

criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight_value, device=device, dtype=torch.float32))
# criterion = FocalLoss(alpha=0.75, gamma=2.0)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=LR_SCHEDULER_FACTOR,
    patience=LR_SCHEDULER_PATIENCE,
)

best_val_loss = float("inf")
best_state = None
patience_counter = 0
history = {"train_loss": [], "val_loss": [], "val_tss": [], "val_hss": [], "val_fpr": [], "val_precision": [], "val_recall": []}
best_threshold = 0.5

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)

    # Validation window-level loss
    model.eval()
    val_loss_sum = 0.0
    with torch.no_grad():
        for batch_x, batch_y, _ in val_loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            logits = model(batch_x)
            loss = criterion(logits, batch_y)
            val_loss_sum += loss.item() * batch_x.size(0)
    val_loss = val_loss_sum / len(val_loader.dataset)

    scheduler.step(val_loss)

    # Tune threshold on sample-level validation with strict FPR constraint
    tune = tune_threshold_sample_level(
        model,
        val_loader,
        device,
        thresholds=THRESHOLD_GRID,
        max_fpr=MAX_ACCEPTABLE_FPR,
        mode=AGGREGATION_MODE,
        percentile=AGGREGATION_PERCENTILE,
    )
    best_threshold = tune["threshold"]
    val_metrics = tune

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_tss"].append(val_metrics["tss"])
    history["val_hss"].append(val_metrics["hss"])
    history["val_fpr"].append(val_metrics["fpr"])
    history["val_precision"].append(val_metrics["precision"])
    history["val_recall"].append(val_metrics["recall"])

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | "
        f"val_tss={val_metrics['tss']:.4f} | val_hss={val_metrics['hss']:.4f} | "
        f"val_fpr={val_metrics['fpr']:.4f} | val_prec={val_metrics['precision']:.4f} | val_rec={val_metrics['recall']:.4f} | "
        f"thr={best_threshold:.3f}"
    )

    # Early stopping based on validation loss
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print("Early stopping triggered.")
            break

if best_state is not None:
    model.load_state_dict(best_state)

print("Best val loss:", best_val_loss)
print("Selected threshold:", best_threshold)

Training windows: pos=151492, neg=159968, pos_weight=1.06
Epoch 01 | train_loss=0.1359 | val_loss=0.1893 | val_tss=0.8707 | val_hss=0.8708 | val_fpr=0.0492 | val_prec=0.9487 | val_rec=0.9199 | thr=0.500
Epoch 02 | train_loss=0.1057 | val_loss=0.1415 | val_tss=0.9073 | val_hss=0.9074 | val_fpr=0.0369 | val_prec=0.9620 | val_rec=0.9442 | thr=0.425
Epoch 03 | train_loss=0.0702 | val_loss=0.0679 | val_tss=0.9600 | val_hss=0.9600 | val_fpr=0.0174 | val_prec=0.9823 | val_rec=0.9774 | thr=0.700
Epoch 04 | train_loss=0.0613 | val_loss=0.0713 | val_tss=0.9543 | val_hss=0.9544 | val_fpr=0.0165 | val_prec=0.9831 | val_rec=0.9708 | thr=0.675
Epoch 05 | train_loss=0.0579 | val_loss=0.0486 | val_tss=0.9682 | val_hss=0.9682 | val_fpr=0.0135 | val_prec=0.9863 | val_rec=0.9817 | thr=0.650
Epoch 06 | train_loss=0.0555 | val_loss=0.1016 | val_tss=0.9389 | val_hss=0.9390 | val_fpr=0.0184 | val_prec=0.9810 | val_rec=0.9573 | thr=0.675
Epoch 07 | train_loss=0.0532 | val_loss=0.0775 | val_tss=0.9488 | val_hs

In [25]:
# Evaluate on validation/test and present metrics as a clean table
import pandas as pd

val_metrics_at_best_threshold = evaluate_sample_level(
    model,
    val_loader,
    device,
    threshold=best_threshold,
    mode=AGGREGATION_MODE,
    percentile=AGGREGATION_PERCENTILE,
)

val_metrics_at_05 = evaluate_sample_level(
    model,
    val_loader,
    device,
    threshold=0.5,
    mode=AGGREGATION_MODE,
    percentile=AGGREGATION_PERCENTILE,
)

test_metrics_at_val_threshold = evaluate_sample_level(
    model,
    test_loader,
    device,
    threshold=best_threshold,
    mode=AGGREGATION_MODE,
    percentile=AGGREGATION_PERCENTILE,
)

best_test_constrained = tune_threshold_sample_level(
    model,
    test_loader,
    device,
    thresholds=THRESHOLD_GRID,
    max_fpr=MAX_ACCEPTABLE_FPR,
    mode=AGGREGATION_MODE,
    percentile=AGGREGATION_PERCENTILE,
)

best_test_unconstrained = tune_threshold_sample_level(
    model,
    test_loader,
    device,
    thresholds=THRESHOLD_GRID,
    max_fpr=None,
    mode=AGGREGATION_MODE,
    percentile=AGGREGATION_PERCENTILE,
)

test_metrics_at_05 = evaluate_sample_level(
    model,
    test_loader,
    device,
    threshold=0.5,
    mode=AGGREGATION_MODE,
    percentile=AGGREGATION_PERCENTILE,
)

rows = [
    ("Validation @ best threshold", val_metrics_at_best_threshold),
    ("Test @ validation threshold", test_metrics_at_val_threshold),
    ("Test best sweep (FPR-constrained)", best_test_constrained),
    ("Validation @ threshold=0.5", val_metrics_at_05),
    ("Test @ threshold=0.5", test_metrics_at_05),
]

table_rows = []
for label, metrics in rows:
    table_rows.append(
        {
            "Split / Setting": label,
            "Threshold": float(metrics["threshold"]),
            "Accuracy": float(metrics["accuracy"]),
            "Precision": float(metrics["precision"]),
            "Recall": float(metrics["recall"]),
            "F1": float(metrics["f1"]),
            "ROC-AUC": float(metrics["roc_auc"]),
            "PR-AUC": float(metrics["pr_auc"]),
            "TSS": float(metrics["tss"]),
            "HSS": float(metrics["hss"]),
        }
    )

metrics_table = pd.DataFrame(table_rows)
metrics_table = metrics_table.round(4)
metrics_table

,Split / Setting,Threshold,Accuracy,Precision,Recall,F1,ROC-AUC,PR-AUC,TSS,HSS
0,Validation @ best threshold,0.725,0.9829,0.9904,0.9751,0.9826,0.9988,0.9989,0.9656,0.9657
1,Test @ validation threshold,0.725,0.9241,0.1579,0.6999,0.2576,0.9372,0.2257,0.6282,0.2341
2,Test best sweep (FPR-constrained),0.875,0.9465,0.1995,0.6115,0.3009,0.9372,0.2257,0.5644,0.2805
3,Validation @ threshold=0.5,0.500,0.9815,0.9735,0.9897,0.9815,0.9988,0.9989,0.9630,0.9629
4,Test @ threshold=0.5,0.500,0.8804,0.1177,0.8237,0.2059,0.9372,0.2257,0.7052,0.1789
